# Lab 3 — Change and uncertainty

**Twenty-five minutes.**

This notebook separates three questions that are often combined in a
curve-number analysis:

1. **Spatial change:** how does the mapped land-cover–soil composition
   change through time?
2. **Event response:** what curve numbers are implied by observed
   rainfall and runoff?
3. **Antecedent state:** how sensitive is the result to the convention
   used to describe conditions before a storm?

The recorded Difficult Run products support the complete analysis. An
optional Earth Engine section repeats the spatial trajectory for a
selected watershed.

Each numbered step states the scientific question, describes the
library operation, shows its intermediate result, and identifies the
interpretation that belongs in a methods or results statement.


In [ ]:
# V3 portable setup: local repository, GitHub Pages bundle, or Colab.
from pathlib import Path
import hashlib
import importlib
import importlib.util
import os
import subprocess
import sys
import urllib.request
import zipfile

CNKIT_VERSION = "1.1.0"
BUNDLE_URL = (
    "https://skp703.github.io/cn-workshop-2026/"
    "downloads/cn_workshop_v3_data.zip"
)
BUNDLE_SHA256 = "925861246fe9520c4b7f399227ca6133e60f27a5063a73c9fb6715cd9904780c"


def _is_workshop_root(path):
    return (path / "data" / "sites.csv").exists() and (path / "prepared").exists()


def _find_workshop_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/cnkit_workshop"),
    ]
    requested = os.environ.get("CNKIT_WORKSHOP_HOME")
    if requested:
        candidates.insert(0, Path(requested).expanduser())
    for candidate in candidates:
        candidate = candidate.resolve()
        if _is_workshop_root(candidate):
            return candidate, "existing workshop folder"

    destination = Path("/content/cnkit_workshop") if Path("/content").exists() else Path.cwd() / ".cnkit_workshop"
    destination.mkdir(parents=True, exist_ok=True)
    archive = destination / "cn_workshop_v3_data.zip"
    print("Downloading the versioned V3 workshop bundle...")
    request = urllib.request.Request(BUNDLE_URL, headers={"User-Agent": "cn-workshop-v3"})
    with urllib.request.urlopen(request, timeout=120) as response, archive.open("wb") as handle:
        handle.write(response.read())
    digest = hashlib.sha256(archive.read_bytes()).hexdigest()
    if digest != BUNDLE_SHA256:
        raise RuntimeError(
            "Workshop bundle checksum mismatch. Expected %s, received %s. "
            "Delete %s and try again." % (BUNDLE_SHA256, digest, archive)
        )
    with zipfile.ZipFile(archive) as zipped:
        zipped.extractall(destination)
    if not _is_workshop_root(destination):
        raise RuntimeError("The workshop bundle downloaded but required files are missing.")
    return destination.resolve(), "checksum-verified workshop download"


WORKSHOP_ROOT, DATA_SOURCE = _find_workshop_root()
DATA_DIR = WORKSHOP_ROOT / "data"
PREPARED_DIR = WORKSHOP_ROOT / "prepared"


def _load_cnkit():
    try:
        import cnkit as package
        if getattr(package, "__version__", None) == CNKIT_VERSION:
            return package, "installed package"
    except ImportError:
        pass

    for candidate in [
        WORKSHOP_ROOT / "vendor" / "cnkit.py",
        WORKSHOP_ROOT / "cnkit.py",
        Path.cwd() / "vendor" / "cnkit.py",
        Path.cwd().parent / "vendor" / "cnkit.py",
    ]:
        if candidate.exists():
            spec = importlib.util.spec_from_file_location("cnkit", candidate)
            package = importlib.util.module_from_spec(spec)
            sys.modules["cnkit"] = package
            spec.loader.exec_module(package)
            return package, str(candidate)

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "cnkit==" + CNKIT_VERSION]
    )
    importlib.invalidate_caches()
    import cnkit as package
    return package, "PyPI"


def activate_full_cnkit():
    """Return the installed package with data, delineation, and GEE modules."""
    global cnkit, CNKIT_SOURCE
    if hasattr(cnkit, "__path__"):
        return cnkit
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "cnkit[gee]==" + CNKIT_VERSION]
    )
    for name in [key for key in sys.modules if key == "cnkit" or key.startswith("cnkit.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
    cnkit = importlib.import_module("cnkit")
    CNKIT_SOURCE = "PyPI with Earth Engine dependencies"
    return cnkit


cnkit, CNKIT_SOURCE = _load_cnkit()
print("cnkit version:", getattr(cnkit, "__version__", CNKIT_VERSION + " workshop module"))
print("cnkit source :", CNKIT_SOURCE)
print("data source  :", DATA_SOURCE)
print("data folder  :", DATA_DIR)
print("setup complete")


In [ ]:
import json
import os
from getpass import getpass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from cnkit import (
    CN_from_PQ,
    compare_conventions,
    doy_climatology,
    fit_asymptotic,
    runoff,
    sm_percentile,
)

USE_EARTH_ENGINE = False
DELINEATION_INPUT = "gage"       # "gage" or "outlet"
GAGE = "01646000"
LAT, LON = 38.97594, -77.24581
YEARS = [2001, 2004, 2007, 2010, 2013, 2016, 2019]
SOILS_SOURCE = "sda"
DESIGN_DEPTH_IN = 4.78


## Part 1 — A spatial curve-number trajectory

This part holds boundary, soil, lookup, condition, scale, and
compositing convention constant while land-cover year changes.


### Step 1 — Define what changes and what remains fixed

For year $t$, the area-weighted curve number is

\[
CN_t=\frac{\sum_i A_{i,t}CN_i}{\sum_i A_{i,t}},
\]

where $A_{i,t}$ is the area of land-cover–soil pair $i$ in year
$t$, and $CN_i$ is the lookup value assigned to that pair. Across
the trajectory, `cnkit` holds the watershed boundary, soil layer,
lookup crosswalk, hydrologic condition, pixel scale, and compositing
convention constant. Annual NLCD is the changing input.

Consequently, the year-to-year difference is attributable to mapped
land-cover change under those fixed analytical choices. It is not a
direct measurement of infiltration, storage, or runoff.


### Step 2 — Load the recorded Earth Engine trajectory

The table below was produced by the live workflow for Difficult Run.
It contains poor, fair, and good hydrologic-condition calculations for
the same annual joint land-cover–soil distribution. The three columns
differ only in the lookup-table condition row.


In [ ]:
trajectory = pd.read_csv(PREPARED_DIR / "difficult_run_gee_trajectory.csv").set_index("year")
recorded = json.loads(
    (PREPARED_DIR / "difficult_run_gee_summary.json").read_text()
)
change = float(trajectory.fair.iloc[-1] - trajectory.fair.iloc[0])
mean_spread = float(trajectory.spread.mean())
ratio = mean_spread / abs(change)

print("boundary method                 NLDI split-catchment upstream")
print("boundary area                   %.4f square miles" % recorded["watershed"]["area_sqmi"])
print("land-cover asset                %s" % recorded["land_cover"]["asset"])
print("soil asset                      %s" % recorded["soils"]["asset"])
print("soil area without mapped HSG    %.2f %%" % recorded["soils"]["percent_area_no_hsg"])
print()
print("2001 CN                         %.4f" % trajectory.fair.iloc[0])
print("2019 CN                         %.4f" % trajectory.fair.iloc[-1])
print("mapped land-cover change       %+.4f CN" % change)
print("mean condition spread           %.4f CN" % mean_spread)
print("condition spread / change        %.1f" % ratio)
display(trajectory.round(4))


### Step 3 — Interpret the hydrologic-condition interval

Poor, fair, and good are field descriptions of cover density,
management, residue, grazing, and related surface conditions. Annual
NLCD classifies land cover but does not observe those attributes.
Recalculating all pixels with the poor and good lookup rows therefore
provides a **sensitivity interval** around the fair-condition result.

This interval is not a statistical confidence interval: no probability
distribution has been assigned to hydrologic condition. Its purpose is
to show how strongly an unobserved table choice affects the trajectory.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.5))

axes[0].plot(
    trajectory.index,
    trajectory.fair,
    "o-",
    color="#007f92",
    lw=2.6,
)
axes[0].set_title("Fair-condition trajectory (expanded scale)")

axes[1].fill_between(
    trajectory.index,
    trajectory.good,
    trajectory.poor,
    color="#c85d45",
    alpha=0.22,
    label="poor-to-good sensitivity interval",
)
axes[1].plot(
    trajectory.index,
    trajectory.fair,
    "o-",
    color="#007f92",
    lw=2.6,
    label="fair-condition trajectory",
)
axes[1].set_title("Trajectory in the condition interval")
axes[1].legend(loc="upper left")

for ax in axes:
    ax.set(xlabel="year", ylabel="composite curve number")
    ax.grid(alpha=0.25)
plt.show()


### Step 4 — Understand how `cn_trajectory` performs the calculation

The convenience workflow organizes the same lower-level operations
used in Lab 2:

1. Accept an existing `Watershed`, or delineate one from coordinates.
2. Create one `Basin` so the boundary and scale remain fixed.
3. For each requested year, call `joint_landcover_soils`; that method
   packs land-cover and soil codes into one integer image, applies one
   Earth Engine frequency histogram, and decodes the observed pairs.
4. Send the same joint table to `composite_from_areas` for poor, fair,
   and good lookup rows. No additional Earth Engine reduction is needed
   for the condition calculations.
5. Optionally transform each composite CN to runoff at a stated design
   depth, then attach assets, scale, boundary, and unmapped-area
   provenance to the returned result.

The workflow is convenient, but the scientific definition of the
trajectory remains the equation in Step 1.


### Step 4A — Delineate a selected watershed

The application is separated into boundary construction and Earth
Engine analysis so each spatial operation can be inspected. Change the
gage or outlet configuration above, then set `USE_EARTH_ENGINE=True`.


In [ ]:
selected_watershed = None

if USE_EARTH_ENGINE:
    activate_full_cnkit()
    from cnkit.delineate import watershed_from_gage, watershed_from_point

    if DELINEATION_INPUT == "gage":
        selected_watershed = watershed_from_gage(GAGE)
    elif DELINEATION_INPUT == "outlet":
        selected_watershed = watershed_from_point(LAT, LON)
    else:
        raise ValueError("DELINEATION_INPUT must be 'gage' or 'outlet'")

    print(selected_watershed)
    print("area: %.3f square miles" % selected_watershed.area_sqmi)
else:
    print("Recorded Difficult Run trajectory selected.")


### Step 4B — Calculate the annual trajectory

Authentication and the annual reductions occur only in this cell. The
preceding boundary can therefore be checked before any raster summary
is requested.


In [ ]:
live_trajectory = None

if USE_EARTH_ENGINE:
    import ee
    from cnkit.gee import initialise
    from cnkit.workflows import cn_trajectory

    project = os.environ.get("CNKIT_EE_PROJECT") or getpass(
        "Earth Engine project ID: "
    )
    ee.Authenticate()
    initialise(project=project)

    live_trajectory = cn_trajectory(
        watershed=selected_watershed,
        project=project,
        years=YEARS,
        condition="fair",
        soils=SOILS_SOURCE,
        design_depth_in=DESIGN_DEPTH_IN,
        progress=lambda done, total, year: print(
            "%d/%d  %d" % (done, total, year)
        ),
    )
    display(live_trajectory)
    print(live_trajectory.summary())
else:
    print("Set USE_EARTH_ENGINE=True to calculate the selected watershed.")


## Part 2 — Curve numbers inferred from observed events

This part changes the evidence base from spatial lookup tables to
rainfall and direct-runoff observations at streamgages.


### Step 5 — Invert the rainfall–runoff equation

For an event with measured rainfall $P$ and direct-runoff depth $Q$,
`CN_from_PQ` solves the curve-number equation backward. With
$I_a=\lambda S$, the physically admissible root is used to recover
$S$, followed by

\[
CN=\frac{1000}{S+10}.
\]

Event CN is therefore a transformed observation, not a direct sensor
measurement. It depends on rainfall, hydrograph separation and runoff
volume, watershed area, event definition, and the selected lambda.
Events with $Q\leq0$, $Q>P$, or no physically valid solution are
excluded from the asymptotic fit.


In [ ]:
difficult_events = pd.read_csv(
    DATA_DIR / "events_01646000.csv", parse_dates=["start", "end"]
)
difficult_events["CN_lambda_020"] = CN_from_PQ(
    difficult_events.P_in.values,
    difficult_events.Q_in.values,
    lam=0.20,
)
difficult_events["CN_lambda_005"] = CN_from_PQ(
    difficult_events.P_in.values,
    difficult_events.Q_in.values,
    lam=0.05,
)
display(
    difficult_events[
        ["start", "P_in", "Q_in", "runoff_ratio", "CN_lambda_020", "CN_lambda_005"]
    ].head(10).round(3)
)
print("event records:", len(difficult_events))


### Step 6 — Estimate the standard asymptotic response

Event-derived CN commonly varies with storm depth. The Hawkins
standard response represents a decreasing sequence that approaches a
stable value as rainfall increases:

\[
CN(P)=CN_{\infty}+(100-CN_{\infty})e^{-kP}.
\]

`fit_asymptotic` first derives event CN with the specified lambda, then
uses bounded nonlinear least squares to estimate $CN_{\infty}$ and
$k$. It returns the fitted parameters, event count, RMSE, and
coefficient of determination. The diagnostic statistics describe this
functional fit; they do not account for uncertainty in precipitation,
discharge, or hydrograph separation.


In [ ]:
fits = []
fitted_objects = {}
for watershed, gage, table_cn in [
    ("Difficult Run", "01646000", 75.5),
    ("Accotink Creek", "01654000", 77.9),
]:
    events = pd.read_csv(DATA_DIR / ("events_" + gage + ".csv"))
    fit20 = fit_asymptotic(events.P_in.values, events.Q_in.values, lam=0.20)
    fit05 = fit_asymptotic(events.P_in.values, events.Q_in.values, lam=0.05)
    fitted_objects[(gage, 0.20)] = fit20
    fitted_objects[(gage, 0.05)] = fit05
    fits.append(
        {
            "watershed": watershed,
            "events": len(events),
            "table_CN": table_cn,
            "CN_inf_lambda_020": fit20.cn_inf,
            "r2_lambda_020": fit20.r2,
            "CN_inf_lambda_005": fit05.cn_inf,
            "r2_lambda_005": fit05.r2,
        }
    )
fit_table = pd.DataFrame(fits).set_index("watershed")
display(fit_table.round(3))


In [ ]:
difficult_fit = fitted_objects[("01646000", 0.20)]
valid_event_cn = difficult_events.replace([np.inf, -np.inf], np.nan).dropna(
    subset=["P_in", "CN_lambda_020"]
)
rainfall_grid = np.linspace(
    valid_event_cn.P_in.min(), valid_event_cn.P_in.max(), 250
)

fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.scatter(
    valid_event_cn.P_in,
    valid_event_cn.CN_lambda_020,
    s=18,
    alpha=0.32,
    color="#6f7f89",
    label="event-derived CN",
)
ax.plot(
    rainfall_grid,
    difficult_fit.predict(rainfall_grid),
    color="#c85d45",
    lw=2.8,
    label=r"standard fit, $CN_{\infty}=%.1f$" % difficult_fit.cn_inf,
)
ax.axhline(75.5, color="#007f92", ls="--", label="table CN = 75.5")
ax.set(xlabel="event rainfall, inches", ylabel="event-derived curve number")
ax.set_ylim(0, 103)
ax.grid(alpha=0.25)
ax.legend()
plt.show()


### Step 7 — Treat lambda and CN as a paired calibration

The tabulated curve numbers were developed with the conventional
relation $I_a=0.20S$. Replacing lambda with 0.05 changes both the
rainfall threshold and the fitted event CN. The two fitted columns
above show why a reported CN must include its lambda; the number and
the equation convention form one calibration.

Compare the two fitted $CN_{\infty}$ values and their diagnostics.
A better fit under one lambda is evidence about this event sample, not
a universal conversion factor for another watershed.


## Part 3 — Antecedent-condition conventions

This part compares two operational descriptions of the watershed state
before an event: five-day rainfall history and seasonally standardized
root-zone wetness.


### Step 8 — Distinguish rainfall history from observed wetness

The historical antecedent moisture condition (AMC) convention assigns
class I, II, or III from five-day rainfall thresholds that vary between
growing and dormant seasons. NEH-630 now uses the broader term
antecedent runoff condition (ARC) to emphasize that runoff response
also reflects cover, temperature, frozen ground, and event history.

The alternative examined here uses NASA POWER `GWETROOT`, a
model-assimilated root-zone wetness index. It represents a broad soil
layer at a comparatively coarse spatial scale; it is not an in-situ
soil-moisture measurement for every point in the watershed.

Raw wetness values have a seasonal cycle. `doy_climatology` pools all
observations within ±15 calendar days of each day of year across the
record. `sm_percentile` compares the previous day's value with that
local seasonal pool. This makes a January and July percentile
comparable while retaining the stated 31-day window as an analytical
choice.


In [ ]:
events = pd.read_csv(DATA_DIR / "events_01646000.csv", parse_dates=["start"])
precipitation = pd.read_csv(
    DATA_DIR / "precip_01646000.csv", parse_dates=["date"]
).set_index("date").P_in
moisture = pd.read_csv(
    DATA_DIR / "soilmoisture_power_01646000.csv", parse_dates=["date"]
).set_index("date").GWETROOT

climatology = doy_climatology(moisture, window=15)
events["previous_day"] = events.start.dt.normalize() - pd.Timedelta(days=1)
events["root_zone_wetness"] = events.previous_day.map(moisture)
events["wetness_percentile"] = [
    sm_percentile(moisture, day, climatology=climatology)
    for day in events.previous_day
]
events["event_cn"] = CN_from_PQ(
    events.P_in.values, events.Q_in.values, lam=0.20
)

display(
    events[
        ["start", "P_in", "Q_in", "root_zone_wetness", "wetness_percentile", "event_cn"]
    ].head(10).round(3)
)


### Step 9 — Compare the conventions on the same storm dates

`compare_conventions` performs both classifications without blending
them. For every event date it:

1. sums the preceding five days of daily rainfall and applies the
   historical seasonal AMC thresholds;
2. calculates the previous-day wetness percentile from the ±15-day
   climatology;
3. maps each class to a CN relative to the same fair-condition `cn2`;
4. optionally applies each CN to the same design storm.

The disagreement rate is a sensitivity diagnostic: it identifies how
often the two proxies point to different antecedent states.


In [ ]:
convention_comparison = compare_conventions(
    events.start,
    precipitation,
    moisture,
    cn2=75.5,
    design_depth_in=DESIGN_DEPTH_IN,
    window=15,
)

print("comparable events:", convention_comparison.attrs["n_comparable"])
print("agreements:       ", convention_comparison.attrs["n_agree"])
print("disagreement rate: %.3f" % convention_comparison.attrs["disagreement_rate"])
display(
    pd.crosstab(
        convention_comparison.AMC_direction,
        convention_comparison.SM_direction,
        margins=True,
    )
)
display(
    convention_comparison.loc[
        ~convention_comparison.conventions_agree,
        [
            "date", "P5_in", "AMC_direction", "SM_percentile",
            "SM_direction", "CN_from_AMC", "CN_from_SM",
            "Q_from_AMC_in", "Q_from_SM_in",
        ],
    ].head(12)
)


### Step 10 — Relate wetness rank to observed event response

The percentile record can also be divided into equal-width lower,
middle, and upper ranges. Grouping observed event CN and runoff ratio
by those ranges tests whether wetter antecedent states correspond to a
systematically different event response in this record. The groups are
descriptive; they do not redefine NRCS ARC classes.


In [ ]:
valid = events.replace([np.inf, -np.inf], np.nan).dropna(
    subset=["wetness_percentile", "event_cn", "runoff_ratio"]
).copy()
valid["wetness_range"] = pd.cut(
    valid.wetness_percentile,
    bins=[0, 33.333, 66.667, 100],
    labels=["lower third", "middle third", "upper third"],
    include_lowest=True,
)
antecedent_summary = valid.groupby("wetness_range", observed=True).agg(
    events=("event_cn", "size"),
    median_percentile=("wetness_percentile", "median"),
    median_event_cn=("event_cn", "median"),
    median_runoff_ratio=("runoff_ratio", "median"),
)
display(antecedent_summary.round(3))


## Method audit — Library operation and analyst decision

| Layer | Operation performed by `cnkit` | Scientific choice retained by the analyst |
|---|---|---|
| `delineate` | Resolves an outlet or gage through NLDI and constructs an upstream boundary | Outlet meaning, delineation route, and boundary verification |
| `gee` | Summarizes annual joint land-cover–soil pixels inside a fixed `Basin` | Imagery year, soil source, scale, and treatment of unmapped area |
| `lookup` | Maps each NLCD–HSG pair to the selected table row and aggregates areas | Crosswalk, hydrologic condition, and composite convention |
| `workflows` | Repeats the same joint calculation across years and records provenance | Which differences are interpreted as temporal change |
| `core` | Evaluates or inverts the rainfall–runoff equation | Lambda, event definition, and measurement basis |
| `asymptotic` | Fits the selected response model to event-derived CN | Model family, event screening, and adequacy of fit |
| `antecedent` | Implements five-day rainfall and wetness-percentile conventions side by side | Proxy, climatology window, thresholds, and interpretation |

The library makes these operations reproducible. It does not select the
scientifically appropriate convention for a particular study.


## Final reporting statement

Write a concise six-part methods-and-results record:

1. Boundary method, area, land-cover source/years, soil source, and
   unmapped fraction.
2. Lookup condition, composite convention, and lambda.
3. Fair-condition trajectory change and poor-to-good sensitivity
   interval, identifying which inputs were held constant.
4. Event-derived asymptotic CN, model form, event count, and fit
   diagnostic for each lambda examined.
5. Antecedent proxies, climatology window, and convention disagreement
   rate.
6. One conclusion that distinguishes mapped change, event response,
   and antecedent-condition sensitivity.

**Source anchors:** USGS NWIS; ACIS/PRISM; NASA POWER; Annual NLCD;
Hawkins (1993); Woodward et al. (2003); NEH-630 Chapters 9 and 10;
NEH-4 Chapter 4 (historical AMC thresholds).
